# 📊 InterpScore — Qwen3.6-27B paper-grade (L11 · L31 · L55)

Companion to [`18_interpscore_eval.ipynb`](./18_interpscore_eval.ipynb), specialized for the 3-layer paper-grade release `caiovicentino1/qwen36-27b-sae-papergrade`.

**What this notebook does**:
1. Loads Qwen3.6-27B once (`AutoModelForImageTextToText` — it's multimodal) in bf16 SDPA
2. Loops over **L11, L31, L55**, loading each SAE
3. Runs the full InterpScore v0.0.1 battery for each layer — `loss_recovered` · `alive` · `l0_score` · `sparse_probing_auc` · `tpp`
4. Uploads per-layer `interpscore_L{N}.json` + a combined `interpscore_papergrade.json` + a comparison chart to the HF SAE repo

**Runtime**: ~90 min on RTX 6000 Pro Blackwell (96 GB) or H100 80 GB. Needs ≥ 80 GB VRAM to fit the base model + one SAE at a time in bf16.

**Why a dedicated notebook?**
- Multimodal loader (`AutoModelForImageTextToText`, not `AutoModelForCausalLM`)
- Hook path is `model.language_model.layers.N`, not `model.layers.N`
- 3 SAEs per run (not one)
- Eval budget scaled for 27B throughput — 250k eval tokens + 75k probe tokens/task keeps each layer ≤ 30 min

Primary sources: Karvonen et al. 2025 ([arXiv:2503.09532](https://arxiv.org/abs/2503.09532)), Gao et al. 2024 ([arXiv:2406.04093](https://arxiv.org/abs/2406.04093)).

In [ ]:
# Always use the latest transformers — Qwen3.6 classes land in 5.x.
# Never pin an old transformers version in this org's notebooks (newer models need newer classes).
!pip install -q -U \
    transformers \
    accelerate \
    safetensors \
    huggingface_hub \
    datasets \
    scikit-learn \
    matplotlib \
    tqdm

import torch, transformers, sklearn
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('sklearn', sklearn.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), 'vram:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Config — paper-grade Qwen3.6-27B

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
LAYERS        = [11, 31, 55]
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128
TOKENS_TRAINED = 200_000_000

# Eval budget — scaled for 27B throughput on RTX 6000 / H100
EVAL_TOKENS      = 250_000
PROBE_TOKENS     = 75_000
PROBING_TASKS    = ['toxicity', 'sentiment']
TPP_CONCEPT      = 'toxicity'
TPP_TOP_FEATURES = 20

# Cache (Google Drive survives kernel death; /tmp does not)
USE_CACHE_DRIVE  = True
CACHE_DIR        = '/content/drive/MyDrive/interpscore_qwen36_27b'

BATCH_TOKENS = 2048   # smaller batch for 27B residual tensors
SEED         = 0
VERSION      = 'v0.0.1'

import os, math, json, time, random
random.seed(SEED); torch.manual_seed(SEED)

if USE_CACHE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        os.makedirs(CACHE_DIR, exist_ok=True)
        print('cache ->', CACHE_DIR)
    except Exception as e:
        print('drive unavailable, using /tmp:', e)
        CACHE_DIR = '/tmp/interpscore_qwen36_27b'
        os.makedirs(CACHE_DIR, exist_ok=True)
else:
    CACHE_DIR = '/tmp/interpscore_qwen36_27b'
    os.makedirs(CACHE_DIR, exist_ok=True)

## 2. Auth + load base model (multimodal loader)

⚠️ **Qwen3.6-27B is multimodal** — must use `AutoModelForImageTextToText`, not `AutoModelForCausalLM`. The residual stream hook path is `model.language_model.layers.N`.

In [ ]:
from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

from transformers import AutoTokenizer, AutoModelForImageTextToText

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map='cuda',
    trust_remote_code=True,
)
model.eval()
print('base model loaded · layers:', len(model.model.language_model.layers))

## 3. SAE wrapper + loader helper

SAEs use the same `W_enc / W_dec / b_enc / b_dec` convention as sae_lens. One SAE at a time — load, eval, free, repeat.

In [ ]:
from safetensors.torch import load_file
import torch.nn.functional as F

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, d_model, d_sae, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16))
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16))
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16))
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16))
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z
    def decode(self, z):
        return z @ self.W_dec + self.b_dec
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z

def load_sae_for_layer(layer):
    path = hf_hub_download(HF_SAE_REPO, f'sae_L{layer}_latest.safetensors')
    sd = load_file(path)
    sae = TopKSAE(sd, D_MODEL, D_SAE, K).to(device).eval()
    return sae

def get_layer_module(layer):
    return model.model.language_model.layers[layer]

## 4. Hooks + eval stream

`capture` grabs the residual after layer N. `replace` swaps it (SAE reconstruction or zero). We use both to compute `loss_recovered`.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

_captured = {}
def capture_hook(mod, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    _captured['resid'] = h
    return out

def replace_hook_factory(new_resid_fn):
    def hook(mod, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        new_h = new_resid_fn(h)
        if isinstance(out, tuple):
            return (new_h,) + out[1:]
        return new_h
    return hook

def stream_batches(ds, batch_tokens=BATCH_TOKENS, total_tokens=EVAL_TOKENS):
    buf, seen = [], 0
    for ex in ds:
        ids = tok(ex['text'], truncation=True, max_length=1024)['input_ids']
        buf.extend(ids)
        while len(buf) >= batch_tokens:
            chunk = buf[:batch_tokens]; buf = buf[batch_tokens:]
            seen += len(chunk)
            yield torch.tensor(chunk, device=device).unsqueeze(0)
            if seen >= total_tokens:
                return

## 5. Per-layer eval function

Bundles core + probing + TPP. Reusable across the 3 layers.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Datasets verified reachable on HF 2026-04-24 with clean binary labels.
# `tasksource/jigsaw_toxicity_pred` is dead — don't use.
PROBING_SOURCES = {
    'toxicity':  ('SetFit/toxic_conversations', None, 'text', 'label'),
    'sentiment': ('sst2', None, 'sentence', 'label'),
}

def load_labelled(task, n=1500):
    try:
        name, cfg, tcol, lcol = PROBING_SOURCES[task]
        ds = load_dataset(name, cfg, split='train', streaming=True) if cfg else load_dataset(name, split='train', streaming=True)
        out = []
        for ex in ds:
            t = ex.get(tcol); l = ex.get(lcol)
            if t is None or l is None: continue
            out.append((t, int(l > 0) if isinstance(l, float) else int(bool(l))))
            if len(out) >= n: break
        return out
    except Exception as e:
        print(f'[warn] {task} failed → synthetic fallback:', e)
        ds = load_dataset('allenai/c4', 'en', split='validation', streaming=True)
        out = []
        for ex in ds:
            out.append((ex['text'][:512], len(ex['text']) % 2))
            if len(out) >= n: break
        return out


def featurise(sae, layer_mod, texts):
    feats = []
    for t in texts:
        ids = tok(t, truncation=True, max_length=256, return_tensors='pt')['input_ids'].to(device)
        with torch.no_grad():
            _captured.clear()
            h = layer_mod.register_forward_hook(capture_hook)
            _ = model(ids)
            h.remove()
            resid = _captured['resid']
            flat = resid.reshape(-1, resid.shape[-1])
            z = sae.encode(flat).float().cpu()
            feats.append(z.mean(dim=0).numpy())
    return np.stack(feats, axis=0)

def featurise_ablated(sae, layer_mod, texts, ablate_idx):
    feats = []
    mask = torch.ones(D_SAE, device=device, dtype=torch.bfloat16)
    mask[ablate_idx] = 0.0
    for t in texts:
        ids = tok(t, truncation=True, max_length=256, return_tensors='pt')['input_ids'].to(device)
        with torch.no_grad():
            _captured.clear()
            h = layer_mod.register_forward_hook(capture_hook)
            _ = model(ids)
            h.remove()
            resid = _captured['resid']
            flat = resid.reshape(-1, resid.shape[-1])
            z = sae.encode(flat) * mask
            feats.append(z.float().cpu().mean(dim=0).numpy())
    return np.stack(feats, axis=0)


def nll(input_ids, mode, sae, layer_mod):
    handles = []
    if mode == 'sae':
        def new_resid(h):
            flat = h.reshape(-1, h.shape[-1])
            recon, _ = sae(flat)
            return recon.reshape(h.shape).to(h.dtype)
        handles.append(layer_mod.register_forward_hook(replace_hook_factory(new_resid)))
    elif mode == 'zero':
        handles.append(layer_mod.register_forward_hook(replace_hook_factory(lambda h: torch.zeros_like(h))))
    with torch.no_grad():
        out = model(input_ids, labels=input_ids)
    for h in handles:
        h.remove()
    return out.loss.item()


def l0_score(L0, target=80):
    if L0 <= 0: return 0.0
    return math.exp(-abs(math.log(L0 / target)))

def evaluate_layer(layer):
    print(f'\n{"="*60}\nEvaluating L{layer}\n{"="*60}')
    t0 = time.time()
    sae = load_sae_for_layer(layer)
    layer_mod = get_layer_module(layer)

    # --- Core metrics ---
    eval_ds = load_dataset('allenai/c4', 'en', split='validation', streaming=True)
    l_clean_sum = l_sae_sum = l_zero_sum = 0.0
    n_batches = 0
    feature_active = torch.zeros(D_SAE, dtype=torch.long, device=device)
    l0_sum = 0.0

    for batch in tqdm(stream_batches(eval_ds), desc=f'L{layer} core', total=EVAL_TOKENS // BATCH_TOKENS):
        l_clean_sum += nll(batch, 'clean', sae, layer_mod)
        with torch.no_grad():
            _captured.clear()
            h = layer_mod.register_forward_hook(capture_hook)
            _ = model(batch)
            h.remove()
        resid = _captured['resid']
        flat = resid.reshape(-1, resid.shape[-1])
        with torch.no_grad():
            z = sae.encode(flat)
        feature_active |= (z.abs() > 0).any(dim=0).long()
        l0_sum += (z != 0).float().sum(dim=-1).mean().item()
        l_sae_sum  += nll(batch, 'sae',  sae, layer_mod)
        l_zero_sum += nll(batch, 'zero', sae, layer_mod)
        n_batches += 1

    L_clean = l_clean_sum / n_batches
    L_sae   = l_sae_sum   / n_batches
    L_zero  = l_zero_sum  / n_batches
    loss_recovered = float(max(0.0, min(1.0, (L_zero - L_sae) / max(L_zero - L_clean, 1e-8))))
    l0 = l0_sum / n_batches
    dead_frac = 1.0 - (feature_active.sum().item() / D_SAE)

    # --- Sparse probing ---
    per_task_auc = {}
    for task in PROBING_TASKS:
        data = load_labelled(task, n=1500)
        random.Random(SEED).shuffle(data)
        split = int(0.8 * len(data))
        tr, te = data[:split], data[split:]
        X_tr = featurise(sae, layer_mod, [x for x,_ in tr]); y_tr = np.array([y for _,y in tr])
        X_te = featurise(sae, layer_mod, [x for x,_ in te]); y_te = np.array([y for _,y in te])
        clf = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced').fit(X_tr, y_tr)
        auc = float(roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1]))
        per_task_auc[task] = auc
        print(f'  L{layer} · {task}: AUROC={auc:.4f}')
    sparse_probing_auc = float(sum(per_task_auc.values()) / len(per_task_auc))

    # --- TPP ---
    data = load_labelled(TPP_CONCEPT, n=1000)
    random.Random(SEED).shuffle(data)
    split = int(0.8 * len(data))
    tr, te = data[:split], data[split:]
    X_tr = featurise(sae, layer_mod, [x for x,_ in tr]); y_tr = np.array([y for _,y in tr])
    X_te = featurise(sae, layer_mod, [x for x,_ in te]); y_te = np.array([y for _,y in te])
    clf = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced').fit(X_tr, y_tr)
    auc_clean = float(roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1]))
    top_feats = np.argsort(-np.abs(clf.coef_[0]))[:TPP_TOP_FEATURES].tolist()
    X_te_ab = featurise_ablated(sae, layer_mod, [x for x,_ in te], top_feats)
    auc_abl = float(roc_auc_score(y_te, clf.predict_proba(X_te_ab)[:, 1]))
    tpp = float(max(0.0, min(1.0, (auc_clean - auc_abl) / max(auc_clean, 1e-8))))
    print(f'  L{layer} · TPP: clean={auc_clean:.4f} ablated={auc_abl:.4f} tpp={tpp:.4f}')

    components = {
        'loss_recovered': loss_recovered,
        'alive':          float(1 - dead_frac),
        'l0_score':       float(l0_score(l0)),
        'sparse_probing': sparse_probing_auc,
        'tpp':            tpp,
    }
    weights = {
        'loss_recovered': 0.30,
        'alive':          0.15,
        'l0_score':       0.15,
        'sparse_probing': 0.25,
        'tpp':            0.15,
    }
    interp_score = float(sum(components[k] * weights[k] for k in weights))

    result = {
        'version': VERSION, 'sae_repo': HF_SAE_REPO, 'model': HF_BASE_MODEL,
        'layer': layer, 'd_model': D_MODEL, 'd_sae': D_SAE, 'k': K,
        'tokens_trained': TOKENS_TRAINED, 'eval_tokens': EVAL_TOKENS,
        'probing_tasks': PROBING_TASKS, 'probing_per_task_auc': per_task_auc,
        'tpp_concept': TPP_CONCEPT, 'tpp_top_features': TPP_TOP_FEATURES,
        'L_clean': L_clean, 'L_sae': L_sae, 'L_zero': L_zero,
        'L0': l0, 'dead_frac': dead_frac,
        'components': components, 'weights': weights, 'interp_score': interp_score,
        'elapsed_sec': time.time() - t0,
    }
    print(f'\n  L{layer} · InterpScore = {interp_score:.4f}  ({time.time()-t0:.0f}s)')

    # Free SAE VRAM before next layer
    del sae
    torch.cuda.empty_cache()
    return result

## 6. Run all three layers

In [ ]:
results = {}
for layer in LAYERS:
    res = evaluate_layer(layer)
    results[layer] = res
    out_path = os.path.join(CACHE_DIR, f'interpscore_L{layer}.json')
    with open(out_path, 'w') as f:
        json.dump(res, f, indent=2)
    print('wrote', out_path)

from datetime import datetime, timezone
combined = {
    'version': VERSION,
    'sae_repo': HF_SAE_REPO,
    'model': HF_BASE_MODEL,
    'tokens_trained': TOKENS_TRAINED,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'per_layer': {str(l): {k: v for k, v in r.items() if k != 'elapsed_sec'} for l, r in results.items()},
    'best_layer': max(results.keys(), key=lambda l: results[l]['interp_score']),
    'sources': {
        'saebench':  'arxiv:2503.09532',
        'gao2024':   'arxiv:2406.04093',
        'neuronpedia': 'https://www.neuronpedia.org/sae-bench',
    },
}
combined_path = os.path.join(CACHE_DIR, 'interpscore_papergrade.json')
with open(combined_path, 'w') as f:
    json.dump(combined, f, indent=2)
print('wrote', combined_path)
print('\nSummary:')
for l, r in results.items():
    print(f'  L{l}: interp_score = {r["interp_score"]:.4f}   '
          f'loss_rec={r["components"]["loss_recovered"]:.3f} '
          f'alive={r["components"]["alive"]:.3f} '
          f'l0={r["components"]["l0_score"]:.3f} '
          f'probe={r["components"]["sparse_probing"]:.3f} '
          f'tpp={r["components"]["tpp"]:.3f}')

## 7. Comparison chart (3 layers side-by-side)

In [ ]:
import matplotlib.pyplot as plt

COLORS = {11: '#8b5cf6', 31: '#f97316', 55: '#16a34a'}
keys = ['loss_recovered', 'alive', 'l0_score', 'sparse_probing', 'tpp']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2), dpi=140)

x = np.arange(len(keys))
w = 0.26
for i, layer in enumerate(LAYERS):
    vals = [results[layer]['components'][k] for k in keys]
    ax1.bar(x + (i-1)*w, vals, width=w, color=COLORS[layer],
            label=f'L{layer}  ({results[layer]["interp_score"]:.4f})',
            edgecolor='white', linewidth=1.2)
ax1.set_xticks(x)
ax1.set_xticklabels(keys, rotation=20, ha='right')
ax1.set_ylim(0, 1.05)
ax1.set_ylabel('Component value')
ax1.set_title('InterpScore v0.0.1 — component breakdown', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right', frameon=False)
ax1.grid(axis='y', color='#f3f4f6', linewidth=1)
for s in ('top', 'right'): ax1.spines[s].set_visible(False)

scores = [results[l]['interp_score'] for l in LAYERS]
bar_colors = [COLORS[l] for l in LAYERS]
bars = ax2.bar([f'L{l}' for l in LAYERS], scores, color=bar_colors,
               edgecolor='white', linewidth=1.5, width=0.6)
for bar, s in zip(bars, scores):
    ax2.annotate(f'{s:.4f}', xy=(bar.get_x()+bar.get_width()/2, s),
                 xytext=(0, 6), textcoords='offset points',
                 ha='center', fontsize=12, fontweight='bold')
ax2.set_ylim(0, max(scores)*1.2)
ax2.set_title('Composite InterpScore per layer', fontsize=12, fontweight='bold')
ax2.set_ylabel('InterpScore v0.0.1')
ax2.grid(axis='y', color='#f3f4f6', linewidth=1)
for s in ('top', 'right'): ax2.spines[s].set_visible(False)

fig.suptitle('Qwen3.6-27B paper-grade SAE · InterpScore per layer',
             fontsize=13.5, fontweight='bold', y=1.02)
fig.text(0.01, 0.0, f'Weights: 0.30 loss_recovered + 0.15 alive + 0.15 l0_score + 0.25 sparse_probing + 0.15 tpp · '
                    f'eval {EVAL_TOKENS//1000}k tokens · probes {PROBING_TASKS} · TPP on {TPP_CONCEPT}',
         fontsize=8, color='#6b7280')
plt.tight_layout()
chart_path = os.path.join(CACHE_DIR, 'interpscore_chart.png')
plt.savefig(chart_path, dpi=140, bbox_inches='tight', facecolor='white')
plt.show()
print('saved', chart_path)

## 8. Upload to HF — per-layer JSON + combined + chart

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

for layer in LAYERS:
    api.upload_file(
        path_or_fileobj=os.path.join(CACHE_DIR, f'interpscore_L{layer}.json'),
        path_in_repo=f'interpscore_L{layer}.json',
        repo_id=HF_SAE_REPO,
        commit_message=f'InterpScore {VERSION} · L{layer} = {results[layer]["interp_score"]:.4f}',
    )

api.upload_file(
    path_or_fileobj=combined_path,
    path_in_repo='interpscore_papergrade.json',
    repo_id=HF_SAE_REPO,
    commit_message=f'InterpScore {VERSION} · combined 3-layer report',
)

api.upload_file(
    path_or_fileobj=chart_path,
    path_in_repo='charts/interpscore_per_layer.png',
    repo_id=HF_SAE_REPO,
    commit_message=f'InterpScore {VERSION} · per-layer chart',
)

print(f'\n✓ uploaded all artifacts to https://huggingface.co/{HF_SAE_REPO}')
print(f'  best layer: L{combined["best_layer"]} with InterpScore {results[combined["best_layer"]]["interp_score"]:.4f}')
print(f'\nNext: PR `openinterpretability-web/lib/leaderboard.ts` with real components.')